# Stage 1 — Canonical Taxonomy & Label Consolidation (labels v3)

Fixes the labelling problems found by `02_data_audit_and_schema_recovery.ipynb`
so the dataset becomes **consistent, high-level, bank-agnostic training data**
for a category + merchant model:

| Problem (from Stage 0 audit) | Fix here |
|---|---|
| 41 overlapping categories (`Loan` / `Home Loan` / `Personal / Business Loan`, `Income` / `Salary`, `Payments` / `Transfer` / `Debit Orders`) | New 2-level taxonomy: 17 canonical categories + subcategories; every old label mapped |
| Status mixed into categories (`Unpaid Insurance`, `Insufficient Funds`, subcategory `Reversal`) | Orthogonal `transaction_status` field (`normal` / `reversal` / `unpaid` / `insufficient_funds`) |
| 10.33M `Miscellaneous/Unknown` rows | Template-majority propagation: unlabelled rows inherit the label of their normalised description template when agreement is high |
| Merchant typos & placeholders (`Old Mutiual`, `nan`, double spaces) | Canonical merchant map + whitespace normalisation |
| Payment *method* labels used as categories (`Debit Orders`) | Dropped as semantic labels (method already lives in `PaymentType`); rows recovered via template propagation |

**Outputs** (all under `ml_model/data/processed/labels_v3/`):
`taxonomy_v3.json`, `template_labels.parquet`,
`transactions_labelled_v3.parquet` (row-level consistent labels + provenance),
`label_quality_report.json` / `.md`.

Deterministic (seed 42), streaming/batched (never loads 37M rows into pandas at
once), read-only on all existing datasets.

In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BATCH_SIZE = 1_048_576
CONSOLIDATE_EVERY = 6            # merge per-batch vote frames every N batches
MIN_TEMPLATE_SUPPORT = 5         # labelled rows needed before a template can vote
MIN_CATEGORY_AGREEMENT = 0.80    # majority share needed to propagate a category
MIN_SUBCAT_AGREEMENT = 0.70
MIN_MERCHANT_AGREEMENT = 0.90
MAX_BATCHES = int(os.environ.get("GALAHAD_LABELS_MAX_BATCHES", "0")) or None


def find_ml_model_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if cand.name == "ml_model" and (cand / "data" / "processed").is_dir():
            return cand
        if (cand / "ml_model" / "data" / "processed").is_dir():
            return cand / "ml_model"
    raise FileNotFoundError(f"ml_model root not found above {start}")


ML_MODEL_ROOT = find_ml_model_root(Path.cwd().resolve())
PROCESSED_DIR = (ML_MODEL_ROOT / "data" / "processed").resolve()
SOURCE_PATH = next(
    p for p in [PROCESSED_DIR / "transactions_for_embeddings.parquet",
                PROCESSED_DIR / "labelled_transactions_v2_regex_corrected.parquet"]
    if p.is_file()
)
OUT_DIR = PROCESSED_DIR / "labels_v3"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PARQUET = OUT_DIR / "transactions_labelled_v3.parquet"
assert os.access(SOURCE_PATH, os.R_OK)

log = logging.getLogger("stage1_labels")
log.setLevel(logging.INFO)
log.handlers.clear()
_fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
for h in (logging.StreamHandler(), logging.FileHandler(OUT_DIR / "labels_run.log", mode="w")):
    h.setFormatter(_fmt)
    log.addHandler(h)
log.info("source: %s", SOURCE_PATH.name)
log.info("outputs: %s", OUT_DIR)
if MAX_BATCHES:
    log.warning("SMOKE-TEST MODE: %d batches only", MAX_BATCHES)

## 1. Canonical taxonomy v3

17 high-level, bank-agnostic categories (+ `uncategorised`), each with
subcategories. Transaction *status* and payment *method* are explicitly **not**
categories — status gets its own field, method already exists as `PaymentType`.

In [ ]:
TAXONOMY_V3 = {
    "version": "3.0",
    "created": datetime.now(timezone.utc).isoformat(),
    "status_values": ["normal", "reversal", "unpaid", "insufficient_funds"],
    "categories": {
        "income": ["salary", "business_income", "pension_benefits",
                   "refunds_reimbursements", "interest_income", "other_income"],
        "transfers": ["send_money", "received_money", "internal_transfer",
                      "wallet_voucher"],
        "cash": ["atm_withdrawal", "cash_deposit", "branch_cash"],
        "bank_fees": ["account_maintenance", "transaction_fees", "penalty_fees",
                      "taxes_duties", "other_bank_charges"],
        "loans_credit": ["personal_loan", "home_loan", "vehicle_finance",
                         "store_credit", "credit_card", "debt_repayment",
                         "microloan"],
        "insurance": ["life_insurance", "funeral_insurance",
                      "short_term_insurance", "medical_aid", "other_insurance"],
        "groceries": ["supermarket", "wholesale", "agricultural_retail",
                      "specialty_food"],
        "food_drink": ["fast_food", "restaurant", "cafe", "bars_nightlife",
                       "alcohol_liquor"],
        "shopping": ["clothing_accessories", "electronics",
                     "furniture_appliances", "building_hardware",
                     "general_retail", "online_shopping"],
        "transport_fuel": ["fuel", "vehicle_maintenance", "public_transport",
                           "car_rental", "parking_tolls"],
        "housing_utilities": ["rent", "municipal_utilities", "electricity_water",
                              "internet_tv", "housing_other"],
        "communication": ["airtime_data", "telecom_contract"],
        "health_beauty": ["pharmacy", "medical", "personal_care", "fitness"],
        "education": ["school_fees", "higher_education", "courses"],
        "entertainment_leisure": ["subscriptions_streaming", "betting_gaming",
                                  "events_travel", "hobbies_outdoors"],
        "savings_investments": ["savings", "unit_trusts",
                                "retirement_contributions", "investments_other"],
        "services_business": ["professional_services", "security_services",
                              "suppliers", "other_services"],
        "uncategorised": [],
    },
}
ALL_CATEGORIES = list(TAXONOMY_V3["categories"])
log.info("taxonomy v3: %d categories, %d subcategories",
         len(ALL_CATEGORIES) - 1,
         sum(len(v) for v in TAXONOMY_V3["categories"].values()))

## 2. Old-label → v3 mappings

Every one of the 41 old categories and 50 old subcategories maps to
`(category_v3, subcategory_v3, status)`. `None` category means "no semantic
information" (unknown/placeholder/method-only labels) — those rows become
candidates for template propagation. Old subcategories come from the regex
rules and are more specific, so they take precedence over old categories;
disagreements between the two are counted as label conflicts.

In [ ]:
# (category_v3, subcategory_v3, status) — None category = no semantic signal
CATEGORY_MAP = {
    "Bank Fees": ("bank_fees", None, None),
    "Miscellaneous/Unknown": (None, None, None),
    "Cash": ("cash", None, None),
    "Insurance": ("insurance", None, None),
    "Personal / Business Loan": ("loans_credit", "personal_loan", None),
    "Payments": ("transfers", None, None),
    "Groceries": ("groceries", None, None),
    "Debit Orders": (None, None, None),          # payment method, not a category
    "Airtime": ("communication", "airtime_data", None),
    "Unpaid Insurance": ("insurance", None, "unpaid"),
    "Loan": ("loans_credit", None, None),
    "Income": ("income", None, None),
    "Shopping": ("shopping", None, None),
    "Health & Beauty": ("health_beauty", None, None),
    "Salary": ("income", "salary", None),
    "General Expenses": (None, None, None),      # too vague to keep
    "Transfer": ("transfers", None, None),
    "Fast Food": ("food_drink", "fast_food", None),
    "Home Loan": ("loans_credit", "home_loan", None),
    "Insufficient Funds": ("bank_fees", "penalty_fees", "insufficient_funds"),
    "Fuel/Retail": ("transport_fuel", "fuel", None),
    "Investments": ("savings_investments", None, None),
    "Savings": ("savings_investments", "savings", None),
    "Utilities": ("housing_utilities", "municipal_utilities", None),
    "Subscriptions": ("entertainment_leisure", "subscriptions_streaming", None),
    "Vehicle and Asset Finance": ("loans_credit", "vehicle_finance", None),
    "Unpaid Vehicle and Asset Finance": ("loans_credit", "vehicle_finance", "unpaid"),
    "Paypulse": ("transfers", "wallet_voucher", None),
    "Housing": ("housing_utilities", None, None),
    "Online Payments": ("shopping", "online_shopping", None),
    "Refund": ("income", "refunds_reimbursements", None),
    "Eating Out": ("food_drink", None, None),
    "Coffee Shops/Cafes": ("food_drink", "cafe", None),
    "Medical Aid": ("insurance", "medical_aid", None),
    "Retirement/Pension Contributions": ("savings_investments", "retirement_contributions", None),
    "Entertainment": ("entertainment_leisure", None, None),
    "Credit Card Payment": ("loans_credit", "credit_card", None),
    "Liquor Stores/Alcohol": ("food_drink", "alcohol_liquor", None),
    "Transport": ("transport_fuel", None, None),
    "Security Services": ("services_business", "security_services", None),
    "Services": ("services_business", None, None),
}

SUBCATEGORY_MAP = {
    "Withdrawal": ("cash", "atm_withdrawal", None),
    "Fees and Taxes": ("bank_fees", None, None),  # too broad for a specific subcat
    "Send Money": ("transfers", "send_money", None),
    "Insurance": ("insurance", None, None),
    "Supermarket": ("groceries", "supermarket", None),
    "Account Maintanance": ("bank_fees", "account_maintenance", None),  # old typo
    "Reversal": (None, None, "reversal"),        # status, not a category
    "Deposit": ("cash", "cash_deposit", None),
    "Airtime": ("communication", "airtime_data", None),
    "Debt Repayment": ("loans_credit", "debt_repayment", None),
    "Clothing & Accessories": ("shopping", "clothing_accessories", None),
    "Fast Food": ("food_drink", "fast_food", None),
    "Personal Care": ("health_beauty", "personal_care", None),
    "Store Account": ("loans_credit", "store_credit", None),
    "Wholesale": ("groceries", "wholesale", None),
    "Vehicle & Asset Finance": ("loans_credit", "vehicle_finance", None),
    "Municipality": ("housing_utilities", "municipal_utilities", None),
    "Personal Loan": ("loans_credit", "personal_loan", None),
    "Agricultural Retail": ("groceries", "agricultural_retail", None),
    "Building Supplies": ("shopping", "building_hardware", None),
    "Pension & Benefits": ("income", "pension_benefits", None),
    "Liquor Store": ("food_drink", "alcohol_liquor", None),
    "Electronics & Home": ("shopping", "electronics", None),
    "Credit Card": ("loans_credit", "credit_card", None),
    "Automotive & Outdoors": ("transport_fuel", "vehicle_maintenance", None),
    "Unit Trusts": ("savings_investments", "unit_trusts", None),
    "Salary": ("income", "salary", None),
    "Fuel": ("transport_fuel", "fuel", None),
    "Savings": ("savings_investments", "savings", None),
    "Vendor": ("services_business", "suppliers", None),
    "Café": ("food_drink", "cafe", None),
    "Furniture & Appliances": ("shopping", "furniture_appliances", None),
    "Restaurant": ("food_drink", "restaurant", None),
    "Supplies": ("services_business", "suppliers", None),
    "Nightlife": ("food_drink", "bars_nightlife", None),
    "Investments": ("savings_investments", None, None),
    "Pharmacy": ("health_beauty", "pharmacy", None),
    "Automotive": ("transport_fuel", "vehicle_maintenance", None),
    "Salary & Wages": ("income", "salary", None),
    "Warehousing": ("services_business", None, None),
    "Doctor Clinic": ("health_beauty", "medical", None),
    "Hardware": ("shopping", "building_hardware", None),
    "Medical Specialist": ("health_beauty", "medical", None),
    "Maintennace & Accessories": ("transport_fuel", "vehicle_maintenance", None),
    "Home Loan": ("loans_credit", "home_loan", None),
    "Tours & Safaris": ("entertainment_leisure", "events_travel", None),
    "Electronic Payment": ("transfers", None, None),
    "Car Rental": ("transport_fuel", "car_rental", None),
    "Higher Education": ("education", "higher_education", None),
    "Hunting & Camping": ("entertainment_leisure", "hobbies_outdoors", None),
}

MERCHANT_FIXES = {
    "Old Mutiual": "Old Mutual",   # 888k rows with typo vs 22k correct
    "nan": None,                   # stringified missing value
}

STATUS_DESC_PATTERNS = [
    ("reversal", re.compile(r"\brevers(?:al|ed|e)\b")),
    ("unpaid", re.compile(r"\bunpaid\b|returned unpaid|\bdishonou?r")),
    ("insufficient_funds", re.compile(r"insufficient funds|\bnsf\b")),
]

# sanity: every mapping target exists in the taxonomy
for m in (CATEGORY_MAP, SUBCATEGORY_MAP):
    for old, (cat, sub, status) in m.items():
        assert cat is None or cat in TAXONOMY_V3["categories"], (old, cat)
        assert sub is None or sub in TAXONOMY_V3["categories"][cat], (old, sub)
        assert status is None or status in TAXONOMY_V3["status_values"]
print(f"mappings validated: {len(CATEGORY_MAP)} categories, "
      f"{len(SUBCATEGORY_MAP)} subcategories, {len(MERCHANT_FIXES)} merchant fixes")

## 3. Row-level assignment logic

Priority order (highest trust first):

1. **`regex_subcategory`** — old subcategory (rule-derived, most specific)
2. **`category_mapping`** — old category mapped to v3
3. **`template_propagation`** — (pass 2 only) majority label of the row's
   normalised description template
4. **`unlabelled`** — stays `uncategorised`

Status is resolved independently from old status-labels *and* description
patterns. The normalisation function is byte-identical to the Stage 0 audit so
template statistics line up across notebooks.

In [ ]:
_norm_digits = re.compile(r"\d+")
_norm_junk = re.compile(r"[^a-z#]+")


def normalise_desc(s: pd.Series) -> pd.Series:
    out = s.astype("string").str.lower()
    out = out.str.replace(_norm_digits, "#", regex=True)
    out = out.str.replace(_norm_junk, " ", regex=True)
    return out.str.strip().str.replace(r"\s+", " ", regex=True)


CAT_TO_V3 = {k: v[0] for k, v in CATEGORY_MAP.items()}
CAT_TO_SUB = {k: v[1] for k, v in CATEGORY_MAP.items()}
CAT_TO_STATUS = {k: v[2] for k, v in CATEGORY_MAP.items()}
SUB_TO_V3 = {k: v[0] for k, v in SUBCATEGORY_MAP.items()}
SUB_TO_SUB = {k: v[1] for k, v in SUBCATEGORY_MAP.items()}
SUB_TO_STATUS = {k: v[2] for k, v in SUBCATEGORY_MAP.items()}


def assign_batch(df: pd.DataFrame) -> pd.DataFrame:
    """Vectorised v3 assignment from old labels (sources 1-2 only)."""
    out = pd.DataFrame(index=df.index)
    desc_lower = df["TransactionParticulars"].astype("string").str.lower().fillna("")

    from_sub = df["subcategory"].map(SUB_TO_V3)
    from_cat = df["Category_corrected"].map(CAT_TO_V3)
    out["category_v3"] = from_sub.combine_first(from_cat)

    sub_from_sub = df["subcategory"].map(SUB_TO_SUB)
    sub_from_cat = df["Category_corrected"].map(CAT_TO_SUB)
    # only keep a subcategory that belongs to the winning category
    out["subcategory_v3"] = sub_from_sub.where(
        from_sub.notna() & (from_sub == out["category_v3"]),
        sub_from_cat.where(from_cat == out["category_v3"]),
    )

    out["label_source"] = np.select(
        [from_sub.notna(), from_cat.notna()],
        ["regex_subcategory", "category_mapping"],
        default="unlabelled",
    )
    out["label_confidence"] = np.select(
        [from_sub.notna(), from_cat.notna()], [0.95, 0.85], default=0.0
    )
    # conflict: both sources give a category and disagree
    out["label_conflict"] = from_sub.notna() & from_cat.notna() & (from_sub != from_cat)

    status = df["subcategory"].map(SUB_TO_STATUS)
    status = status.combine_first(df["Category_corrected"].map(CAT_TO_STATUS))
    for status_name, pattern in STATUS_DESC_PATTERNS:
        hit = desc_lower.str.contains(pattern, regex=True)
        status = status.where(~(status.isna() & hit), status_name)
    out["transaction_status"] = status.fillna("normal")

    merchant = df["merchant_name"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    merchant = merchant.replace(MERCHANT_FIXES)
    out["merchant_v3"] = merchant
    out["merchant_source"] = np.where(merchant.notna(), "regex_rule", None)
    return out

print("assignment logic ready")

## 4. Pass 1 — template vote aggregation (streaming)

Aggregates, per normalised description template: labelled-row category votes,
subcategory votes and merchant votes. Vote frames are consolidated every few
batches so memory stays bounded.

In [ ]:
pf = pq.ParquetFile(SOURCE_PATH)
VOTE_COLS = ["TransactionParticulars", "Category_corrected", "subcategory", "merchant_name"]

cat_votes, sub_votes, merch_votes, tmpl_totals = [], [], [], []
conflict_pairs: Counter = Counter()
n_conflicts = 0
rows_pass1 = 0
t0 = time.time()


def consolidate(frames: list[pd.DataFrame], keys: list[str]) -> list[pd.DataFrame]:
    if not frames:
        return frames
    merged = pd.concat(frames).groupby(keys, sort=False)["n"].sum().reset_index()
    return [merged]


for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE, columns=VOTE_COLS)):
    if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
        break
    df = batch.to_pandas()
    rows_pass1 += len(df)
    assigned = assign_batch(df)

    # map raw description -> template via its per-batch uniques (cheap)
    raw = df["TransactionParticulars"].astype("string").fillna("")
    uniq = pd.Series(raw.unique())
    tmpl_map = dict(zip(uniq, normalise_desc(uniq)))
    tmpl = raw.map(tmpl_map)

    n_conflicts += int(assigned["label_conflict"].sum())
    conflict_pairs.update(
        df.loc[assigned["label_conflict"], ["subcategory", "Category_corrected"]]
          .astype(str).agg(" | ".join, axis=1).value_counts().to_dict()
    )

    work = pd.DataFrame({"template": tmpl,
                         "cat": assigned["category_v3"],
                         "sub": assigned["subcategory_v3"],
                         "merchant": assigned["merchant_v3"]})
    tmpl_totals.append(work.groupby("template", sort=False).size().rename("n").reset_index())
    labelled = work.dropna(subset=["cat"])
    cat_votes.append(labelled.groupby(["template", "cat"], sort=False).size().rename("n").reset_index())
    with_sub = labelled.dropna(subset=["sub"])
    sub_votes.append(with_sub.groupby(["template", "cat", "sub"], sort=False).size().rename("n").reset_index())
    with_merch = work.dropna(subset=["merchant"])
    merch_votes.append(with_merch.groupby(["template", "merchant"], sort=False).size().rename("n").reset_index())

    if (batch_idx + 1) % CONSOLIDATE_EVERY == 0:
        cat_votes = consolidate(cat_votes, ["template", "cat"])
        sub_votes = consolidate(sub_votes, ["template", "cat", "sub"])
        merch_votes = consolidate(merch_votes, ["template", "merchant"])
        tmpl_totals = consolidate(tmpl_totals, ["template"])
        log.info("pass1 batch %d — %s rows (%.0fs)", batch_idx + 1,
                 f"{rows_pass1:,}", time.time() - t0)

cat_votes = consolidate(cat_votes, ["template", "cat"])[0]
sub_votes = consolidate(sub_votes, ["template", "cat", "sub"])[0]
merch_votes = consolidate(merch_votes, ["template", "merchant"])[0]
tmpl_totals = consolidate(tmpl_totals, ["template"])[0]
log.info("pass1 done: %s rows, %s templates, %s (template,cat) pairs, "
         "%s row-level conflicts (%.0fs)",
         f"{rows_pass1:,}", f"{len(tmpl_totals):,}", f"{len(cat_votes):,}",
         f"{n_conflicts:,}", time.time() - t0)

## 5. Resolve template-level labels

A template earns a propagatable label when it has enough labelled support and a
clear majority. Generic templates (e.g. bare `#`) fail the agreement threshold
and stay unresolved — that is the desired behaviour.

In [ ]:
votes_tot = cat_votes.groupby("template")["n"].sum().rename("labelled_n")
top_cat = (cat_votes.sort_values("n", ascending=False)
           .drop_duplicates("template").set_index("template"))
tmpl = pd.DataFrame({"labelled_n": votes_tot}).join(top_cat[["cat", "n"]])
tmpl["agreement"] = tmpl["n"] / tmpl["labelled_n"]
tmpl = tmpl.join(tmpl_totals.set_index("template")["n"].rename("total_rows"))

# guard: refuse to propagate from generic templates (no real word left after
# normalisation, e.g. bare '#' from pure-number descriptions)
has_real_word = tmpl.index.to_series().str.contains(r"[a-z]{3,}", regex=True)
resolved = tmpl[(tmpl["labelled_n"] >= MIN_TEMPLATE_SUPPORT)
                & (tmpl["agreement"] >= MIN_CATEGORY_AGREEMENT)
                & has_real_word].copy()

# subcategory: majority *within* the winning category
sub_top = (sub_votes.merge(resolved.reset_index()[["template", "cat"]],
                           on=["template", "cat"])
           .sort_values("n", ascending=False).drop_duplicates("template"))
sub_tot = sub_votes.groupby("template")["n"].sum().rename("sub_n")
sub_top = sub_top.set_index("template").join(sub_tot)
sub_top = sub_top[sub_top["n"] / sub_top["sub_n"] >= MIN_SUBCAT_AGREEMENT]
resolved["sub"] = sub_top["sub"]

# merchant: strict majority over all rows that carried a merchant
merch_tot = merch_votes.groupby("template")["n"].sum().rename("merch_n")
merch_top = (merch_votes.sort_values("n", ascending=False)
             .drop_duplicates("template").set_index("template").join(merch_tot))
merch_top = merch_top[(merch_top["merch_n"] >= MIN_TEMPLATE_SUPPORT)
                      & (merch_top["n"] / merch_top["merch_n"] >= MIN_MERCHANT_AGREEMENT)]

template_labels = resolved.reset_index().rename(columns={
    "cat": "category_v3", "sub": "subcategory_v3"})
template_labels["confidence"] = (template_labels["agreement"] * 0.85).round(3)
template_labels = template_labels[["template", "category_v3", "subcategory_v3",
                                   "confidence", "agreement", "labelled_n", "total_rows"]]
merchant_labels = merch_top.reset_index()[["template", "merchant"]] \
    .rename(columns={"merchant": "merchant_v3"})

template_labels.to_parquet(OUT_DIR / "template_labels.parquet", index=False)
merchant_labels.to_parquet(OUT_DIR / "template_merchants.parquet", index=False)

ambiguous_n = int(((tmpl["labelled_n"] >= MIN_TEMPLATE_SUPPORT)
                   & (tmpl["agreement"] < MIN_CATEGORY_AGREEMENT)).sum())
log.info("templates: %s total | %s category-resolved | %s ambiguous | %s merchant-resolved",
         f"{len(tmpl):,}", f"{len(template_labels):,}", f"{ambiguous_n:,}",
         f"{len(merchant_labels):,}")

TMPL_CAT = dict(zip(template_labels["template"], template_labels["category_v3"]))
TMPL_SUB = dict(zip(template_labels["template"],
                    template_labels["subcategory_v3"].where(
                        template_labels["subcategory_v3"].notna(), None)))
TMPL_CONF = dict(zip(template_labels["template"], template_labels["confidence"]))
TMPL_MERCH = dict(zip(merchant_labels["template"], merchant_labels["merchant_v3"]))
template_labels.sort_values("total_rows", ascending=False).head(15)

## 6. Pass 2 — write `transactions_labelled_v3.parquet`

Streams the source again, applies row-level assignment, fills unlabelled rows
from resolved templates, and writes batch-by-batch with `ParquetWriter`. The
`template` column is kept so training can deduplicate/stratify by template.

In [ ]:
KEEP_COLS = ["Unnamed: 0", "TransactionDate", "TransactionParticulars",
             "TransactionAmount", "TransactionTypeDebitCredit", "PaymentType",
             "Channel", "ChannelCode", "Category_corrected", "subcategory",
             "merchant_name", "matched_rule_id"]
NEW_COLS = ["template", "category_v3", "subcategory_v3", "merchant_v3",
            "transaction_status", "label_source", "label_confidence", "merchant_source"]

writer = None
counts = {"rows": 0, "propagated_cat": 0, "propagated_merch": 0}
source_counter: Counter = Counter()
cat_counter: Counter = Counter()
status_counter: Counter = Counter()
merch_before = merch_after = 0
t0 = time.time()

for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE)):
    if MAX_BATCHES is not None and batch_idx >= MAX_BATCHES:
        break
    df = batch.to_pandas()
    assigned = assign_batch(df)

    raw = df["TransactionParticulars"].astype("string").fillna("")
    uniq = pd.Series(raw.unique())
    tmpl_map = dict(zip(uniq, normalise_desc(uniq)))
    assigned["template"] = raw.map(tmpl_map)

    # template propagation for rows with no semantic label
    unlab = assigned["category_v3"].isna()
    prop_cat = assigned.loc[unlab, "template"].map(TMPL_CAT)
    hit = prop_cat.notna()
    idx = prop_cat.index[hit]
    assigned.loc[idx, "category_v3"] = prop_cat[hit]
    assigned.loc[idx, "subcategory_v3"] = assigned.loc[idx, "template"].map(
        lambda t: TMPL_SUB.get(t))
    assigned.loc[idx, "label_source"] = "template_propagation"
    assigned.loc[idx, "label_confidence"] = assigned.loc[idx, "template"].map(TMPL_CONF)
    counts["propagated_cat"] += int(hit.sum())

    merch_before += int(assigned["merchant_v3"].notna().sum())
    no_merch = assigned["merchant_v3"].isna()
    prop_merch = assigned.loc[no_merch, "template"].map(TMPL_MERCH)
    mhit = prop_merch.notna()
    midx = prop_merch.index[mhit]
    assigned.loc[midx, "merchant_v3"] = prop_merch[mhit]
    assigned.loc[midx, "merchant_source"] = "template_propagation"
    counts["propagated_merch"] += int(mhit.sum())
    merch_after += int(assigned["merchant_v3"].notna().sum())

    assigned["category_v3"] = assigned["category_v3"].fillna("uncategorised")

    out_df = pd.concat([df[KEEP_COLS].rename(columns={"Unnamed: 0": "row_id"}),
                        assigned[NEW_COLS]], axis=1)
    table = pa.Table.from_pandas(out_df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_PARQUET, table.schema, compression="snappy")
    else:
        table = table.cast(writer.schema)
    writer.write_table(table)

    counts["rows"] += len(out_df)
    source_counter.update(assigned["label_source"].value_counts().to_dict())
    cat_counter.update(assigned["category_v3"].value_counts().to_dict())
    status_counter.update(assigned["transaction_status"].value_counts().to_dict())
    if (batch_idx + 1) % CONSOLIDATE_EVERY == 0:
        log.info("pass2 batch %d — %s rows (%.0fs)", batch_idx + 1,
                 f"{counts['rows']:,}", time.time() - t0)

if writer is not None:
    writer.close()

out_meta = pq.ParquetFile(OUTPUT_PARQUET).metadata
assert out_meta.num_rows == counts["rows"], "output row count mismatch"
log.info("pass2 done: wrote %s rows -> %s (%.0f MB, %.0fs)",
         f"{out_meta.num_rows:,}", OUTPUT_PARQUET.name,
         OUTPUT_PARQUET.stat().st_size / 1e6, time.time() - t0)

## 7. Label quality report — before vs after

In [ ]:
total = counts["rows"]
labelled_after = total - cat_counter.get("uncategorised", 0)
cat_dist = pd.Series(cat_counter).sort_values(ascending=False)
source_dist = pd.Series(source_counter).sort_values(ascending=False)

quality_report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "source_dataset": str(SOURCE_PATH),
    "output_dataset": str(OUTPUT_PARQUET),
    "rows": total,
    "smoke_test_mode": MAX_BATCHES is not None,
    "taxonomy_version": TAXONOMY_V3["version"],
    "category_coverage": {
        "before_usable_rate": 0.7221,   # Stage 0 audit, Category_corrected excl. placeholders
        "after_labelled_rows": int(labelled_after),
        "after_labelled_rate": round(labelled_after / total, 4),
        "recovered_by_template_propagation": counts["propagated_cat"],
    },
    "merchant_coverage": {
        "before_rate": 0.5986,          # Stage 0 audit
        "after_rows": int(merch_after),
        "after_rate": round(merch_after / total, 4),
        "recovered_by_template_propagation": counts["propagated_merch"],
    },
    "label_sources": {k: int(v) for k, v in source_dist.items()},
    "category_distribution_v3": {k: int(v) for k, v in cat_dist.items()},
    "transaction_status_distribution": {k: int(v) for k, v in status_counter.items()},
    "row_level_conflicts_subcat_vs_category": int(n_conflicts),
    "top_conflict_pairs": [
        {"subcategory | old_category": k, "rows": int(v)}
        for k, v in conflict_pairs.most_common(15)
    ],
    "templates": {
        "total": int(len(tmpl)),
        "category_resolved": int(len(template_labels)),
        "ambiguous_high_support": ambiguous_n,
        "merchant_resolved": int(len(merchant_labels)),
        "thresholds": {
            "min_support": MIN_TEMPLATE_SUPPORT,
            "min_category_agreement": MIN_CATEGORY_AGREEMENT,
            "min_subcategory_agreement": MIN_SUBCAT_AGREEMENT,
            "min_merchant_agreement": MIN_MERCHANT_AGREEMENT,
        },
    },
}


def save_json(obj, path):
    def default(o):
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        return str(o)
    Path(path).write_text(json.dumps(obj, indent=2, default=default))
    log.info("wrote %s", Path(path).name)


save_json(TAXONOMY_V3, OUT_DIR / "taxonomy_v3.json")
save_json(quality_report, OUT_DIR / "label_quality_report.json")

report_md = [
    "# Label consolidation report (labels v3)",
    f"- generated: {quality_report['generated_at']}",
    f"- rows: {total:,}",
    f"- category coverage: 72.2% -> "
    f"{quality_report['category_coverage']['after_labelled_rate']:.1%} "
    f"(+{counts['propagated_cat']:,} rows via template propagation)",
    f"- merchant coverage: 59.9% -> "
    f"{quality_report['merchant_coverage']['after_rate']:.1%} "
    f"(+{counts['propagated_merch']:,} rows via template propagation)",
    f"- row-level label conflicts (regex subcat vs category): {n_conflicts:,}",
    "",
    "## v3 category distribution",
] + [f"- {k}: {v:,}" for k, v in cat_dist.items()] + [
    "", "## Status distribution",
] + [f"- {k}: {v:,}" for k, v in pd.Series(status_counter).sort_values(ascending=False).items()]
(OUT_DIR / "label_quality_report.md").write_text("\n".join(report_md))

print(f"category coverage:  72.2% -> {labelled_after / total:.1%}")
print(f"merchant coverage:  59.9% -> {merch_after / total:.1%}")
print(f"status != normal:   {total - status_counter.get('normal', 0):,} rows")
print()
print(cat_dist.to_string())
print()
print(source_dist.to_string())

## 8. Result

`labels_v3/` now contains a consistent, hierarchical, bank-agnostic labelled
dataset:

- **`transactions_labelled_v3.parquet`** — every row carries `category_v3`
  (17 canonical categories or `uncategorised`), `subcategory_v3`,
  `merchant_v3` (typo-fixed, propagated), `transaction_status`, the
  description `template`, and full provenance (`label_source`,
  `label_confidence`, `merchant_source`).
- **`taxonomy_v3.json`** — the canonical taxonomy + status values, versioned,
  for the app to map old labels forward.
- **`template_labels.parquet` / `template_merchants.parquet`** — reusable
  template-level knowledge base (also useful as a fast rule-like classifier at
  inference time).
- **`label_quality_report.json|.md`** — before/after coverage, conflict
  counts, distributions.

Next: `04_train_category_merchant_model.ipynb` trains the hybrid
encoder + category head + merchant retrieval model on this dataset.